# MagicTensors Demo

`TODO: Write descriptive text for this MagicTensors demo.`

In [1]:
using MagicTensors

In [2]:
N = 5;

## Pauli Strings

In [4]:
QubitPauli"-iIXYZ"

-i_XYZ

In [4]:
embed(QubitPauli"-iXYZ", 10, [5,2,10])

-i_Y__X____Z

In [34]:
QubitPauli"XX" + QubitPauli"YY"

QubitPauliSum{ComplexF64, Vector{UInt64}} on 2 sites, with 2 terms:
(+1.000000e+00 +0.000000e+00im) + YY
(+1.000000e+00 +0.000000e+00im) + XX

In [6]:
QubitPauli"XX" + QubitPauli"-iXX"

QubitPauliSum{ComplexF64, Vector{UInt64}} on 2 sites, with 1 terms:
(+1.000000e+00 -1.000000e+00im) + XX

In [7]:
# exp(iϕP) as a `QubitPauliSum`
exp_pauli(P::QubitPauli, ϕ::Float64) = cos(ϕ)*QubitPauli(nsites(P)) + 1.0im*sin(ϕ)*P
exp_pauli(QubitPauli"Z", π/8)

QubitPauliSum{ComplexF64, Vector{UInt64}} on 1 sites, with 2 terms:
(+9.238795e-01 +0.000000e+00im) + _
(+0.000000e+00 +3.826834e-01im) + Z

### 1D Cluster State Hamiltonian
H = − X₁ Z₂ − ∑ᵢ Zᵢ₋₁ Xᵢ Zᵢ₊₁ − Zₙ₋₁ Xₙ

In [8]:
H = QubitPauliSum(N)
push!(H, QubitPauli"XZ", [1,2], -1)
for i∈2:N-1
    push!(H, QubitPauli"ZXZ", [i-1,i,i+1], -1)
end
push!(H, QubitPauli"ZX", [N-1,N], -1)
H

QubitPauliSum{ComplexF64, Vector{UInt64}} on 5 sites, with 5 terms:
(-1.000000e+00 +0.000000e+00im) + _ZXZ_
(-1.000000e+00 +0.000000e+00im) + __ZXZ
(-1.000000e+00 +0.000000e+00im) + ZXZ__
(-1.000000e+00 +0.000000e+00im) + ___ZX
(-1.000000e+00 +0.000000e+00im) + XZ___

## Cliffords

### Clifford Gate

In [10]:
import QuantumClifford

In [11]:
Hadamard = QubitCliffordGate(QuantumClifford.tHadamard)

QubitCliffordGate acting on 1 qubits:
X₁ ⟼ + Z
Z₁ ⟼ + X

In [12]:
CNOT = QubitCliffordGate(QuantumClifford.tCNOT)

QubitCliffordGate acting on 2 qubits:
X₁ ⟼ + XX
X₂ ⟼ + _X
Z₁ ⟼ + Z_
Z₂ ⟼ + ZZ

In [15]:
Matrix(CNOT)

StackOverflowError: StackOverflowError:

### CliffordGateSet

In [13]:
gateset = QubitCliffordGateSet(2, :entangle)

QubitCliffordGateSet{2, :entangle}()

In [14]:
nsites(gateset), length(gateset)

(2, 19)

In [15]:
gateset[2]

QubitCliffordGate acting on 2 qubits:
X₁ ⟼ + XX
X₂ ⟼ + _X
Z₁ ⟼ + Z_
Z₂ ⟼ + ZZ

### Clifford Unitary

In [16]:
C = QubitCliffordUnitary(N)
apply!(C, Hadamard, [2])
apply!(C, CNOT, [2,3])

QubitCliffordUnitary acting on 5 qubits:
X₁ ⟼ + X____
X₂ ⟼ + _Z___
X₃ ⟼ + __X__
X₄ ⟼ + ___X_
X₅ ⟼ + ____X
Z₁ ⟼ + Z____
Z₂ ⟼ + _XX__
Z₃ ⟼ + _ZZ__
Z₄ ⟼ + ___Z_
Z₅ ⟼ + ____Z

In [17]:
inv(C)

QubitCliffordUnitary acting on 5 qubits:
X₁ ⟼ + X____
X₂ ⟼ + _ZX__
X₃ ⟼ + __X__
X₄ ⟼ + ___X_
X₅ ⟼ + ____X
Z₁ ⟼ + Z____
Z₂ ⟼ + _X___
Z₃ ⟼ + _XZ__
Z₄ ⟼ + ___Z_
Z₅ ⟼ + ____Z

In [18]:
conjugate(H, C)   # C H C†

QubitPauliSum{ComplexF64, Vector{UInt64}} on 5 sites, with 5 terms:
(-1.000000e+00 +0.000000e+00im) + _X_Z_
(-1.000000e+00 +0.000000e+00im) + _ZZXZ
(-1.000000e+00 +0.000000e+00im) + Z_Z__
(-1.000000e+00 +0.000000e+00im) + ___ZX
(-1.000000e+00 +0.000000e+00im) + XXX__

## ITensorMPS.MPS

In [5]:
import ITensorMPS

In [6]:
mps = ITensorMPS.MPS(ITensorMPS.siteinds("Qubit", N), "0");
expectation(mps, QubitPauli"XX", [2,3])

0.000

In [7]:
apply!(mps, exp_pauli(QubitPauli"Y",-π/8), [2])
apply!(mps, CNOT, [2,3])
expectation(mps, QubitPauli"XX", [2,3])

LoadError: UndefVarError: `exp_pauli` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [22]:
entanglement_entropy(mps)

4-element Vector{Float64}:
 6.406853007629834e-16
 0.6008760366928565
 6.406853007629834e-16
 6.406853007629834e-16

In [37]:
stabilizer_entropy(mps; α=1.0)
# stabilizer_entropy(mps, 1000; α=1.0)

1.000

In [24]:
mps

5-element ITensorMPS.MPS:
 ((dim=2|id=422|"Qubit,Site,n=1"), (dim=1|id=965|"Link,l=1"))
 ((dim=2|id=25|"Qubit,Site,n=2"), (dim=1|id=965|"Link,l=1"), (dim=2|id=83|"Link,u"))
 ((dim=2|id=83|"Link,u"), (dim=2|id=486|"Qubit,Site,n=3"), (dim=1|id=444|"Link,l=3"))
 ((dim=2|id=788|"Qubit,Site,n=4"), (dim=1|id=205|"Link,l=4"), (dim=1|id=444|"Link,l=3"))
 ((dim=2|id=713|"Qubit,Site,n=5"), (dim=1|id=205|"Link,l=4"))

## CAMPS

In [25]:
camps = QubitCAMPS(mps)

QubitCAMPS, |ψ⟩ = C |MPS⟩, with
  |MPS⟩ with bond dimensions:
    [1, 2, 1, 1]
  C:
    QubitCliffordUnitary acting on 5 qubits:
    X₁ ⟼ + X____
    X₂ ⟼ + _X___
    X₃ ⟼ + __X__
    X₄ ⟼ + ___X_
    X₅ ⟼ + ____X
    Z₁ ⟼ + Z____
    Z₂ ⟼ + _Z___
    Z₃ ⟼ + __Z__
    Z₄ ⟼ + ___Z_
    Z₅ ⟼ + ____Z

In [38]:
disentangler = GreedySweepDisentangler(QubitCliffordGateSet(2,:entangle); cutoff=1e-12)
disentangle!(disentangler, camps)

(4, 4.000, Tuple{Int64, Float64, Any}[(1, 1.000, 4), (1, 1.000, 4), (1, 1.000, 4), (1, 1.000, 4)])

In [30]:
camps

QubitCAMPS, |ψ⟩ = C |MPS⟩, with
  |MPS⟩ with bond dimensions:
    [1, 1, 1, 1]
  C†:
    QubitCliffordUnitary acting on 5 qubits:
    X₁ ⟼ + ZX___
    X₂ ⟼ + XZX__
    X₃ ⟼ + _XZX_
    X₄ ⟼ + __XZZ
    X₅ ⟼ + ___XX
    Z₁ ⟼ + X____
    Z₂ ⟼ + _X___
    Z₃ ⟼ + __X__
    Z₄ ⟼ + ___X_
    Z₅ ⟼ + ____Z

## DMRG

In [39]:
camps = QubitCAMPS(N)
# energy, _ = dmrg!(camps, H; nsweeps=1)
energy, _ = dmrg!(TrivialDisentangler(), camps, H; nsweeps=1)
@show energy
camps

After sweep 1 energy=-5.000000000000002  maxlinkdim=2 maxerr=5.82E-62 time=0.005
energy = -5.000


QubitCAMPS, |ψ⟩ = C |MPS⟩, with
  |MPS⟩ with bond dimensions:
    [2, 2, 2, 2]
  C†:
    QubitCliffordUnitary acting on 5 qubits:
    X₁ ⟼ + X____
    X₂ ⟼ + _X___
    X₃ ⟼ + __X__
    X₄ ⟼ + ___X_
    X₅ ⟼ + ____X
    Z₁ ⟼ + Z____
    Z₂ ⟼ + _Z___
    Z₃ ⟼ + __Z__
    Z₄ ⟼ + ___Z_
    Z₅ ⟼ + ____Z